In [ ]:
import time

_run_start = time.perf_counter()

# Waterkwaliteit NiFi pipeline

Load and validate the NiFi definition (with its deployment overlay), then compile `:DemonstratorPipeline` and write the build to `out/nifi`. The cell layout mirrors `demo_fietsstallingen.ipynb` so the per-cell timings are comparable.

In [ ]:
import json
from pathlib import Path

from rdflib import Graph
from rdfine import GraphReader
from compilers import (
    CompilationConfig,
    CompilationRunner,
    FileMaterializer,
    PipelineGeneratorConfig,
)

In [ ]:
data_dir = Path("../data")
source_files = [
    "catalog/catalog-core.ttl",
    "catalog/catalog-ldio.ttl",
    "catalog/catalog-nifi.ttl",
    "catalog/catalog-rdfc.ttl",
    "catalog/catalog-rdfc-manual.ttl",
    "catalog/catalog-sw.ttl",
    "pipelines/pipeline_definition_nifi.ttl",
    "pipelines/pipeline_definition_nifi.deployment.ttl",
    "catalog/catalog-application-profile-shapes.ttl",
]

graph = Graph()
for filename in source_files:
    graph.parse(data_dir / filename, publicID="file:///workspace/pipeline/")

reader = GraphReader(graph)
for rules in ("inference_rules/inference_rules.yaml", "inference_rules/rdfc_inference_rules.yaml"):
    reader = reader.infer(data_dir / rules)

In [ ]:
report = reader.validate(advanced=True, inference="rdfs")
violations = report.select(
    "?focus ?message",
    """
    ?result a sh:ValidationResult ;
        sh:focusNode ?focus ;
        sh:resultMessage ?message .
    """,
)

if not report.ask("?report sh:conforms true"):
    for row in violations.itertuples(index=False):
        print(f"{row.focus}: {row.message}")
    raise ValueError("NiFi definition does not conform")

print("The NiFi definition conforms.")

Compile the single plan with the generator's compiler list, fed with the graph loaded and validated above.

In [ ]:
config = CompilationConfig(compilers=PipelineGeneratorConfig.compilers, graph=reader.graph)

generator = CompilationRunner(":DemonstratorPipeline", config)
build_graph = generator.compile()
builder = FileMaterializer(build_graph)

print(", ".join(compiler.__name__ for compiler in config.compilers))

In [ ]:
for _, file in builder.files.iterrows():
    content = file["content"]
    if file["filename"] == "flow.json":
        flow = json.loads(content)
        for kind in ("processors", "controllerServices"):
            for component in flow["rootGroup"][kind]:
                for name, descriptor in component["propertyDescriptors"].items():
                    if descriptor["sensitive"] and name in component["properties"]:
                        component["properties"][name] = "[REDACTED]"
        content = json.dumps(flow, indent=4)

    print(f"=== {file['filepath']}/{file['filename']} ===")
    print(content)

Materialize the build below `out/nifi/`. Existing generated files at the same paths are overwritten by `FileMaterializer`.

In [ ]:
written = builder.write("../out/nifi")

for path in written:
    print(path)

In [ ]:
_elapsed = time.perf_counter() - _run_start
print(f"Total generation runtime: {_elapsed:.2f} s")